## Investigación exhaustiva

¡Uno de los casos de uso clásicos de Agentic en diversos negocios! Esto es importantísimo.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Implicaciòn comenrcial</h2>
            <span style="color:#00bfff;">Un agente de Investigación Profunda es ampliamente aplicable a cualquier área de negocio y a tus actividades diarias. ¡Puedes usarlo tú mismo!
            </span>
        </td>
    </tr>
</table>

In [ ]:

# Importaciones principales del framework del curso para crear el agente y rastrear su ejecución.
# NOTA: 'WebSearchTool' es la que genera el cobro de 2.5 centavos. Más adelante la interceptaremos o reemplazaremos.
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
# Esta clase es CRUCIAL. Aquí es donde cambiaremos la configuración para que apunte 
# a Groq, Gemini o tu Ollama local en lugar de los servidores de OpenAI.
from agents.model_settings import ModelSettings
# Pydantic nos sirve para estructurar los datos que el agente va a extraer de su investigación.
from pydantic import BaseModel, Field
from dotenv import load_dotenv
# Permite que el agente ejecute tareas asíncronas (como buscar en la web mientras procesa texto).
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
# Tipado de datos para estructurar las respuestas en los diccionarios de Python.
from typing import Dict
# Permite que los reportes finales de la investigación se vean estéticos y formateados dentro de Cursor/Jupyter.
from IPython.display import display, Markdown

In [2]:
load_dotenv(override=True)

True

## Herramientas alojadas de OpenAI

El SDK de agentes de OpenAI incluye las siguientes herramientas alojadas:

La herramienta `WebSearchTool` permite a un agente buscar en la web.

La herramienta `FileSearchTool` permite recuperar información de los almacenes de vectores de OpenAI.

La herramienta `ComputerTool` permite automatizar tareas de uso del ordenador, como tomar capturas de pantalla y hacer clic.

### Nota importante: Costo de la API de WebSearchTool

El costo de cada llamada a OpenAI WebSearchTool es de 2.5 centavos. Esto puede sumar entre $2 y $3 para los próximos dos laboratorios. Usaremos herramientas de búsqueda gratuitas y de bajo costo con otras plataformas, así que si el costo les preocupa, pueden omitir esta parte. Además, el estudiante Christian W. señaló que OpenAI a veces cobra por varias búsquedas en una sola llamada, por lo que el costo podría superar los 2.5 centavos por llamada.

Costs are here: https://platform.openai.com/docs/pricing#web-search

In [ ]:
INSTRUCTIONS = "Eres un asistente de investigación. Dado un término de búsqueda, buscas en la web ese término y \
produces un resumen conciso de los resultados. El resumen debe tener entre 2 y 3 párrafos y menos de 300 palabras. \
Captura los puntos principales. Escribe de manera sucinta, no es necesario tener oraciones completas ni buena gramática.\
Esto será consumido por alguien que está sintetizando un reporte, por lo que es vital que captures la esencia e ignores cualquier relleno.\
No incluyas ningún comentario adicional que no sea el resumen en sí."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,

    tools=[WebSearchTool(search_context_size="low")],

    model="gpt-4o-mini",
    
    model_settings=ModelSettings(tool_choice="required"),
)

In [ ]:
message = "Ultimos frameworks de agentes de IA en 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

### As always, take a look at the trace

https://platform.openai.com/traces

###Ahora utilizaremos Salidas Estructuradas e incluiremos una descripción de los campos.

In [ ]:
# Consulte la nota anterior sobre el costo de WebSearchTool.

HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"Eres un asistente de investigación muy útil. Dado un termino de busqueda,\
 propone una conjunto de búsquedas  web para realizar y así obtener la mejor respuesta posible. \
  Salida {HOW_MANY_SEARCHES} tèrminos para consultar."

# Usa Pydantic para definir el esquema de nuestra respuesta; esto se conoce como "Salidas Estructuradas".

class WebSearchItem(BaseModel):
    reason: str = Field(description="Su razonamiento sobre por qué esta búsqueda es importante para la consulta..")

    query: str = Field(description="El término de búsqueda que se utilizará para la búsqueda web.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="Una lista de búsquedas web para realizar y obtener la mejor respuesta a la consulta..")


planner_agent = Agent(
    name="Agente de planificaciòn",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [ ]:

message = "Ultimos frameworks de agentes de IA en 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Envía un correo electrónico con el asunto y el cuerpo HTML indicados. """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("ed@edwarddonner.com") # Change this to your verified email
    to_email = To("ed.donner@gmail.com") # Change this to your email
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [ ]:
send_email

In [ ]:
INSTRUCTIONS = """Puedes enviar un correo electrónico HTML con un formato atractivo a partir de un informe detallado.

Recibirás un informe detallado. Debes usar la herramienta para enviar un correo electrónico, 
con el informe convertido a HTML limpio y bien presentado, con un asunto apropiado."""

email_agent = Agent(
    name="Agente de correo electrònico",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)



In [ ]:
INSTRUCTIONS = (
"Eres un investigador sénior encargado de redactar un informe coherente para una consulta de investigación."
"Se te proporcionará la consulta original y una investigación inicial realizada por un asistente de investigación.\n"
"Primero, debes elaborar un esquema del informe que describa su estructura y"
"flujo. Luego, genera el informe y entrégalo como resultado final."
"El resultado final debe estar en formato Markdown y debe ser extenso y detallado."
"El objetivo es que tenga entre 5 y 10 páginas de contenido, con al menos 1000 palabras."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="Un resumen de 2-3 pàrrafos de los resultados.")

    markdown_report: str = Field(description="El informe final")

    follow_up_questions: list[str] = Field(description="Temas sigeridos para investigar màs")


writer_agent = Agent(
    name="Agente de escritura",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

### Las siguientes 3 funciones planificara y ejecutara la busqueda, uyilizando planner_agent and search_agent

In [ ]:
async def plan_searches(query: str):
    """ Utilice planner_agente para planificar què bùsquedas ejecutar parala consulta """
    print("Planificando bùsquedas...")
    result = await Runner.run(planner_agent, f"Consulta: {query}")
    print(f"Se realizaràn {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Llama a search() para cada elemento en el plan de busqueda """
    print("Buscando...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Busqueda finalizada")
    return results

async def search(item: WebSearchItem):
    """ Usa el agente de bùsqueda para ejecutar una bùsqueda web paracada elemento en el plan de bùsqueda """
    input = f"Termino de busqueda: {item.query}\nRazòn para buscar: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### Las siguientes 2 funciones escriben un informe y lo envìan por correro electònico

In [ ]:
async def write_report(query: str, search_results: list[str]):
    """ Usa el agente de escritura para escribir un informe basado en los resultados de la bùsqueda"""
    print("Pensando sobre el informe...")
    input = f"Consulta original: {query}\nResultados de busqueda resumidos: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Informe finalizado")
    return result.final_output

async def send_email(report: ReportData):
    """ Usa el agente de correo electronico para enviar un correo electronico con el informe """
    print("Escribiendo correo electrònico...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Correo electònico enviado")
    return report

Hora del espectàculo

### Showtime!

In [ ]:
query ="Ultimos frameworks de agentes de IA en 2025"

with trace("Investigacion"):
    print("Iniciando Investigaciòn...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Felicidades!")




### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Felicitaciones por su progreso y una solicitud</h2>
            <span style="color:#00cc00;">Has llegado a un momento importante del curso; has creado un Agente valioso utilizando uno de los frameworks de agentes más recientes. Has mejorado tus habilidades y desbloqueado nuevas posibilidades comerciales. ¡Tómate un momento para celebrar tu éxito!<br/><br/>Algo que debería preguntarte — mi editor me golpearía si no mencionara esto. Si puedes calificar el curso en Udemy, te estaría muy agradecido: es la forma más importante en que Udemy decide si mostrar el curso a otras personas y marca una gran diferencia.<br/><br/>Y otro recordatorio para <a href="https://www.linkedin.com/in/eddonner/">conectar conmigo en LinkedIn</a> si lo deseas. Si quisieras publicar sobre tu progreso en el curso, por favor mencióname y yo intervendré para aumentar tu visibilidad.
            </span>
        </td>
    </tr>